# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx
from skewerjax.constants.physical_constants import phys

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import Skewer
from skewerjax.cosmology.cosmology import Cosmology

In [ ]:
# Standard Planck 2015 flat ΛCDM cosmology
cosmo = Cosmology(
    H0=phys.H0,
    Omega_r_0=9.1436e-5,  # Radiation density (CMB + neutrinos)
    Omega_m_0=0.3089,
    Omega_k_0=0.0,
    Omega_lambda_0=0.6911,
)

In [ ]:
# ============================================================
# 2. Create individual skewers with new API
# ============================================================

L_box = 1.0  # Mpc/h

# Skewer 1: 5 points along x-axis at y=0, z=0, spanning 4*L_box
# Default: a0_defined_at_index=0 (a0 at first point, left-to-right direction)
spatial_points_1 = L_box * jnp.array([
    [0.0, 0.0, 0.0],   # x=0
    [0.1, 0.0, 0.0],   # x=0.1*L_box
    [0.2, 0.0, 0.0],   # x=0.2*L_box
    [0.3, 0.0, 0.0],   # x=0.3*L_box
    [0.4, 0.0, 0.0],   # x=0.4*L_box
])  # shape: (5, 3), range: 0 to 40 Mpc/h

skewer_1 = Skewer(spatial_points=spatial_points_1, a0=1.0)
print("Skewer 1 created (default: a0 at first point):")
print(f"  N_ppsk: {skewer_1.N_ppsk}")
print(f"  a0_defined_at_index: {skewer_1.a0_defined_at_index}")
print(f"  Auto-set direction: {skewer_1.index_direction}")
print(f"  Comoving distances: {skewer_1.compute_comoving_distances()}")
print()

# Skewer 2: 3 points along diagonal, up to L_box*sqrt(3)
# Demonstrates that direction is now auto-set, no manual specification needed
spatial_points_2 = L_box * jnp.array([
    [0.0, 0.0, 0.0],
    [0.5, 0.5, 0.5],   # x=y=z=L_box/2
    [1.0, 1.0, 1.0],   # x=y=z=L_box*sqrt(3)
])  # shape: (3, 3)

# Same as skewer_1: a0 at start, left-to-right by default
skewer_2 = Skewer(spatial_points=spatial_points_2, a0=1.0)
print("Skewer 2 created (diagonal configuration):")
print(f"  N_ppsk: {skewer_2.N_ppsk}")
print(f"  a0_defined_at_index: {skewer_2.a0_defined_at_index}")
print(f"  Auto-set direction: {skewer_2.index_direction}")
print(f"  LoS directions (local): {skewer_2.compute_los_directions()}")
print()

# Skewer 3: 7 points with a0 at the END (reverse integration)
# This now uses a0_defined_at_index=-1 instead of a0_at_start=False
spatial_points_3 = (6.0 * L_box) * jnp.linspace(0, 1, 7).reshape(-1, 1).repeat(3, axis=1)

skewer_3 = Skewer(
    spatial_points=spatial_points_3, 
    a0=0.5,  # Scale factor at z=1, defined at the last point
    a0_defined_at_index=-1  # New API: -1 means a0 at last point
)
print("Skewer 3 created (a0 at END, z=1):")
print(f"  N_ppsk: {skewer_3.N_ppsk}")
print(f"  a0_defined_at_index: {skewer_3.a0_defined_at_index}")
print(f"  Auto-set direction: {skewer_3.index_direction}")
print(f"  Comoving distances: {skewer_3.compute_comoving_distances()}")
print()

In [ ]:
# ============================================================
# 3. Demonstrate cosmology integration
# ============================================================

# Compute scale factors for each skewer
print("Scale factors along skewers:")
print(f"  Skewer 1: {skewer_1.compute_scale_factors(cosmo)}")
print(f"  Skewer 2: {skewer_2.compute_scale_factors(cosmo)}")
print(f"  Skewer 3: {skewer_3.compute_scale_factors(cosmo)}")
print()

In [ ]:
# Show LoS directions for each skewer
print("Line-of-sight directions (first 3 points):")
print(f"  Skewer 1: {skewer_1.compute_los_directions()[:3]}")
print(f"  Skewer 2: {skewer_2.compute_los_directions()[:3]}")
print(f"  Skewer 3: {skewer_3.compute_los_directions()[:3]}")
print()